# 06. Relationship Analysis: How Are Variables Related?
### Primary Analytical Purpose: *How are two (or more) variables related?*
**Lecture Reference:** Purpose 6 — Relationship (Slides 29–32, 45)

Relationship analysis investigates whether and how two or more numeric variables move together. It evaluates:
- **Direction:** Is the relationship positive (both rise together) or negative (one rises as the other falls)?
- **Strength:** How tightly do observations cluster around the pattern line?
- **Form:** Is the association linear, curved, or clustered into distinct subsets?
- **Outliers & Influential Points:** Are there extreme points deviating from the overall pattern?

---

## 1. Scenario and Dataset Preview
**Business Scenario:** A Doha retail chain evaluates 16 consecutive weekly marketing campaigns. The dataset tracks:
- `ad_spend_k_qar`: Weekly digital advertising spend (in thousands of QAR)
- `revenue_k_qar`: Weekly store revenue (in thousands of QAR)
- `conversions`: Number of successful online checkouts
- `impressions_k`: Ad impressions (thousands)
- `roas`: Return on Ad Spend (`revenue / ad_spend`)

The dataset is loaded from `data/qatar_digital_marketing.csv`. Let us inspect the records.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# Set modern Seaborn theme
sns.set_theme(style="whitegrid")

# Load marketing dataset
df = pd.read_csv("data/qatar_digital_marketing.csv")

# Display the dataset
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df

## 2. Primary Visualization: Scatter Plot with Trendline (The Default)
- **What the chart shows:** Weekly campaigns plotted as points with the presumed driver (`ad_spend_k_qar`) on the x-axis and outcome (`revenue_k_qar`) on the y-axis, fitted with an ordinary least squares (OLS) regression line.
- **When it is useful:** Default choice for examining the relationship between two numeric variables (Slide 29).
- **Critical Rule from Lecture (Slide 30 & 32):**
  - **'Associated with' is defensible; 'causes' is not.** Correlation does not prove causation!
  - Always word chart titles honestly (e.g. 'Higher ad spend is associated with higher revenue', not 'Ad spend drives revenue').

In [ ]:
# Calculate Pearson correlation coefficient
corr_coef = df["ad_spend_k_qar"].corr(df["revenue_k_qar"])

# --- Seaborn Implementation: Scatter Plot with Regression Line ---
plt.figure(figsize=(8.5, 4.8))

ax = sns.regplot(
    data=df, 
    x="ad_spend_k_qar", 
    y="revenue_k_qar", 
    color="#D64545",  # Trendline in dark red
    scatter_kws={"s": 65, "color": "#14A098"},  # Points in teal
    ci=None  # Keep plot clean for beginners
)

# Principle 4: Honest zero baselines
ax.set_xlim(0, 55)
ax.set_ylim(0, 110)

# Principle 2: Title states association honestly; labels carry units
ax.set_title(f"Higher Ad Spend Is Associated with Higher Revenue (r = {corr_coef:.2f})", fontsize=13, weight="bold")
ax.set_xlabel("Weekly Ad Spend (k QAR)", fontsize=11)
ax.set_ylabel("Weekly Revenue (k QAR)", fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# --- Plotly Express Implementation: Interactive Scatter with OLS Trendline ---
fig = px.scatter(
    df, 
    x="ad_spend_k_qar", 
    y="revenue_k_qar", 
    trendline="ols",
    hover_name="week",
    hover_data=["conversions", "roas"],
    title=f"Higher Ad Spend Is Associated with Higher Revenue (r = {corr_coef:.2f})",
    labels={"ad_spend_k_qar": "Weekly Ad Spend (k QAR)", "revenue_k_qar": "Weekly Revenue (k QAR)"}
)

fig.update_traces(marker=dict(size=10, color="#14A098"))
fig.update_xaxes(range=[0, 55])
fig.update_yaxes(range=[0, 110])
fig.show()

### Structured Interpretation: The 5-Step Framework
1. **Question:** Is weekly digital ad spend correlated with revenue in Doha stores, and how strong is the association?
2. **Visualization:** Scatter plot with fitted OLS linear regression line over observed spend range (8k–48k QAR).
3. **Observation:** A very strong positive linear association exists ($r = 0.98$). Every additional 10k QAR in weekly spend is associated with ~15.5k QAR in additional revenue.
4. **Insight:** Higher-spend weeks consistently produce higher revenue across the 8k–48k QAR range. However, correlation alone does not prove that ad spend caused the entire sales bump.
5. **Implication:** The relationship is strong over the tested range; run a controlled split-region test before expanding weekly marketing budgets above 50k QAR.

## 3. Alternative 1: Bubble Chart (Encoding a 3rd Variable)
- **What the chart shows:** A scatter plot where marker size encodes a third numeric variable (checkout conversions) and marker color encodes ROAS.
- **When to use instead:** When a third continuous dimension is essential to understand transaction volume (Slide 29 & 45).
- **Caution (Slide 29):** Use sparingly; human vision decodes circular area far less accurately than position.

In [ ]:
# --- Plotly Express: Bubble Chart ---
fig_bubble = px.scatter(
    df, 
    x="ad_spend_k_qar", 
    y="revenue_k_qar", 
    size="conversions",
    color="roas",
    hover_name="week",
    size_max=18,
    color_continuous_scale="Viridis",
    title="Bubble Chart: Conversion Volume and ROAS Across Campaign Weeks",
    labels={
        "ad_spend_k_qar": "Weekly Ad Spend (k QAR)",
        "revenue_k_qar": "Weekly Revenue (k QAR)",
        "conversions": "Conversions",
        "roas": "ROAS"
    }
)

fig_bubble.update_xaxes(range=[0, 55])
fig_bubble.update_yaxes(range=[0, 110])
fig_bubble.show()

### Interpretation for Bubble Chart
- **Observation:** Bubble sizes grow proportionally along the diagonal. Campaigns in Weeks 12–16 generated over 700 conversions each.
- **Derived Insight:** Higher ad budgets not only associate with top-line revenue but also drive higher total conversion volumes, maintaining healthy overall ROAS.

## 4. Alternative 2: Correlation Heatmap (Multi-Variable Screening)
- **What the chart shows:** A grid displaying pairwise Pearson correlation coefficients ($r$) between all numeric variables, colored with a diverging scale.
- **When to use instead:** As a high-level screening tool across many variables before generating detailed scatter plots (Slide 29 & 45).

In [ ]:
# Compute correlation matrix for numeric metrics
numeric_cols = ["ad_spend_k_qar", "revenue_k_qar", "conversions", "impressions_k", "roas"]
corr_matrix = df[numeric_cols].corr()

# --- Seaborn Implementation: Correlation Heatmap ---
plt.figure(figsize=(7.5, 5))

sns.heatmap(
    corr_matrix, 
    annot=True, 
    fmt=".2f", 
    cmap="vlag",  # Diverging colormap centered at zero
    vmin=-1, 
    vmax=1, 
    linewidths=0.5,
    cbar_kws={"label": "Pearson Correlation (r)"}
)

plt.title("Correlation Matrix: Ad Spend, Revenue, and Conversions Move Together", fontsize=12, weight="bold")
plt.tight_layout()
plt.show()

### Interpretation for Heatmap
- **Observation:** `ad_spend_k_qar`, `revenue_k_qar`, `conversions`, and `impressions_k` all show near-perfect mutual correlation ($r > 0.95$). Meanwhile, `roas` shows a slightly negative correlation ($r = -0.44$) with ad spend.
- **Derived Insight:** As ad spend expands to higher levels, marginal returns slightly soften (typical diminishing returns in advertising).

## 5. Best Practices & Common Mistakes (Lecture Principles)
**Visualization Best Practices (Slide 31 & 46):**
- **Plot the data before computing $r$:** Anscombe's quartet shows that identical correlation values can hide completely different shapes.
- **Put the presumed driver on the x-axis** and the outcome metric on the y-axis.
- **Use transparency (alpha) or jitter** when dealing with hundreds or thousands of overlapping points.
- **State the unit of observation:** Here, one point represents one week of business operations.

**Common Mistakes & Misleading Designs:**
- **Claiming causation from a scatter plot alone:** Correlation only proves co-movement, not causality.
- **Extrapolating beyond the observed range (Slide 30 & 31):** Assuming that spending 200k QAR will yield 350k QAR revenue is invalid without empirical evidence at that scale.
- **Ignoring lurking variables:** A holiday festival could cause both ad spend and revenue to surge simultaneously.

## 6. Key Takeaways
1. **Scatter plots are the primary tool for bivariate relationships**, revealing direction, strength, linearity, and outliers.
2. **Correlation is not causation:** Always phrase titles cautiously ('associated with' rather than 'causes').
3. **Driver on the x-axis, outcome on the y-axis:** Respect standard causal orientation.
4. **Heatmaps serve as screening mechanisms;** scatter plots provide the detailed evidence.